# Chapter 5 Student Colab Notebook: Inner Product Spaces

**Course:** MATH 3260 - Linear Algebra  
**Sections:** 5.1-5.5  
**Audience:** Student version

This notebook is the complete interactive teaching companion for Chapter 5. It combines theory, original worked examples, detailed calculations, geometric interpretation, applications, Python visualizations, interactive demonstrations, section worksheets, and complete instructor solutions.

> **How to use this student notebook**
>
> Run the cells from top to bottom in Google Colab. Read the complete theory and worked examples, explore the visualizations, and complete each section worksheet. Complete instructor solutions are not included. Display mathematics uses `$$...$$` for reliable Colab rendering.

## Chapter map

1. **Section 5.1:** Length and Dot Product in $\mathbb{R}^n$  
2. **Section 5.2:** Inner Product Spaces  
3. **Section 5.3:** Orthonormal Bases and the Gram-Schmidt Process  
4. **Section 5.4:** Mathematical Models and Least Squares Analysis  
5. **Section 5.5:** Applications of Inner Product Spaces  

Each section ends with a terminology table, a worksheet, and complete instructor solutions.

In [ ]:
# Core imports used throughout Chapter 5
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from IPython.display import display, Math, Markdown
from mpl_toolkits.mplot3d import Axes3D

try:
    from ipywidgets import interact, FloatSlider, IntSlider
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

sp.init_printing()

def show_matrix(M, label=None):
    M = sp.Matrix(M)
    if label:
        display(Math(rf"{label}={sp.latex(M)}"))
    else:
        display(M)
    return M

def show_vector(v, label=None):
    v = sp.Matrix(v)
    if label:
        display(Math(rf"{label}={sp.latex(v)}"))
    else:
        display(v)
    return v

def show_exact(expr, label):
    display(Math(rf"{label}={sp.latex(sp.simplify(expr))}"))
    return sp.simplify(expr)

def plot_vectors_2d(vectors, labels, title='Vectors in the plane'):
    fig, ax = plt.subplots(figsize=(6.5,5.5))
    maxval = max(max(abs(float(x)) for x in v) for v in vectors) + 1
    for v,label in zip(vectors,labels):
        ax.quiver(0,0,float(v[0]),float(v[1]),angles='xy',scale_units='xy',scale=1)
        ax.text(float(v[0]),float(v[1]),f'  {label}')
    ax.axhline(0,linewidth=.8); ax.axvline(0,linewidth=.8)
    ax.set_xlim(-maxval,maxval); ax.set_ylim(-maxval,maxval)
    ax.set_aspect('equal'); ax.grid(alpha=.3); ax.set_title(title)
    plt.show()

def plot_vectors_3d(vectors, labels, title='Vectors in space'):
    fig = plt.figure(figsize=(7,6))
    ax = fig.add_subplot(111, projection='3d')
    maxval = max(max(abs(float(x)) for x in v) for v in vectors) + 1
    for v,label in zip(vectors,labels):
        ax.quiver(0,0,0,float(v[0]),float(v[1]),float(v[2]),arrow_length_ratio=.08)
        ax.text(float(v[0]),float(v[1]),float(v[2]),f'  {label}')
    ax.set_xlim(-maxval,maxval); ax.set_ylim(-maxval,maxval); ax.set_zlim(-maxval,maxval)
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z'); ax.set_title(title)
    plt.show()

def projection(u, v):
    u, v = sp.Matrix(u), sp.Matrix(v)
    return sp.simplify((u.dot(v)/v.dot(v))*v)

def gram_schmidt_detailed(vectors):
    ws=[]
    for j,v in enumerate([sp.Matrix(x) for x in vectors], start=1):
        display(Markdown(f'**Construct $\\mathbf{{w}}_{j}$**'))
        display(Math(rf"\mathbf{{v}}_{j}={sp.latex(v)}"))
        w=v
        for i,prev in enumerate(ws, start=1):
            coeff=sp.simplify(v.dot(prev)/prev.dot(prev))
            display(Math(rf"\operatorname{{proj}}_{{\mathbf{{w}}_{i}}}(\mathbf{{v}}_{j})={sp.latex(coeff)}\mathbf{{w}}_{i}"))
            w=sp.simplify(w-coeff*prev)
        ws.append(sp.Matrix(w))
        display(Math(rf"\mathbf{{w}}_{j}={sp.latex(ws[-1])}"))
    us=[]
    for i,w in enumerate(ws, start=1):
        norm=sp.sqrt(w.dot(w))
        u=sp.simplify(w/norm)
        us.append(u)
        display(Math(rf"\mathbf{{u}}_{i}=\frac{{\mathbf{{w}}_{i}}}{{\|\mathbf{{w}}_{i}\|}}={sp.latex(u)}"))
    return ws,us

> **Original-example policy**
>
> All worked examples and numerical values are original. Themes are used selectively from the San Antonio Spurs, hiking, badminton, driving, travel, Backstreet Boys, and R&B music. Any prices, routes, coordinates, listening data, or performance data are fictional teaching models unless explicitly identified otherwise.

# Section 5.1 - Length and Dot Product in $\mathbb{R}^n$

## Learning objectives

Students should be able to:

- compute vector length and construct unit vectors;
- compute distances between vectors;
- use the dot product and its algebraic properties;
- find angles and test orthogonality;
- apply Cauchy-Schwarz, the triangle inequality, and the Pythagorean theorem;
- express the dot product as a matrix product.

### Definition: norm

> For $\mathbf{v}=(v_1,\ldots,v_n)$, the Euclidean norm is $\|\mathbf{v}\|=\sqrt{v_1^2+\cdots+v_n^2}$. A vector with norm $1$ is a unit vector.

## Spurs-inspired norm example

Let

$$
\mathbf{v}=(9,12,20).
$$

Then

$$
\|\mathbf{v}\|
=\sqrt{9^2+12^2+20^2}
=\sqrt{81+144+400}
=\sqrt{625}
=25.
$$

The unit vector in the direction of $\mathbf{v}$ is

$$
\widehat{\mathbf{v}}
=\frac{\mathbf{v}}{\|\mathbf{v}\|}
=\left(\frac9{25},\frac{12}{25},\frac45\right).
$$

In [ ]:
v = sp.Matrix([9,12,20])
norm_v = sp.sqrt(v.dot(v))
show_exact(norm_v, r'\|\mathbf{v}\|')
show_vector(sp.simplify(v/norm_v), r'\widehat{\mathbf{v}}')
print('Unit-vector check:', sp.simplify((v/norm_v).dot(v/norm_v)))

> **Distance**
>
> The Euclidean distance between $\mathbf{u}$ and $\mathbf{v}$ is $d(\mathbf{u},\mathbf{v})=\|\mathbf{u}-\mathbf{v}\|$.

## Travel-coordinate distance

Suppose two fictional travel profiles are encoded by

$$
\mathbf{u}=(3,-2,5),
\qquad
\mathbf{v}=(9,1,-3).
$$

Then

$$
\mathbf{u}-\mathbf{v}=(-6,-3,8),
$$

so

$$
d(\mathbf{u},\mathbf{v})
=\sqrt{(-6)^2+(-3)^2+8^2}
=\sqrt{109}.
$$

### Definition: dot product

> For $\mathbf{u},\mathbf{v}\in\mathbb{R}^n$, $\mathbf{u}\cdot\mathbf{v}=u_1v_1+\cdots+u_nv_n$.

## Dot product and angle

Let

$$
\mathbf{a}=(4,1,-2),
\qquad
\mathbf{b}=(2,-3,5).
$$

The dot product is

$$
\mathbf{a}\cdot\mathbf{b}
=4(2)+1(-3)+(-2)(5)
=-5.
$$

Also,

$$
\|\mathbf{a}\|=\sqrt{21},
\qquad
\|\mathbf{b}\|=\sqrt{38}.
$$

Therefore,

$$
\cos\theta
=\frac{\mathbf{a}\cdot\mathbf{b}}{\|\mathbf{a}\|\|\mathbf{b}\|}
=-\frac5{\sqrt{798}}.
$$

For nonzero vectors,

$$
\cos\theta=\frac{\mathbf a\cdot\mathbf b}{\|\mathbf a\|\,\|\mathbf b\|}.
$$

Thus

$$
\mathbf a\perp\mathbf b
\iff
\mathbf a\cdot\mathbf b=0,
$$

which corresponds to a $90^\circ$ angle.


In [ ]:
a=sp.Matrix([4,1,-2]); b=sp.Matrix([2,-3,5])
cos_theta=sp.simplify(a.dot(b)/(sp.sqrt(a.dot(a))*sp.sqrt(b.dot(b))))
theta=sp.acos(cos_theta)
show_exact(cos_theta,r'\cos\theta')
print('Angle in degrees:',float(sp.N(theta*180/sp.pi,6)))
plot_vectors_3d([a,b],['a','b'],'Angle between two vectors')

> **Orthogonality**
>
> Two vectors are orthogonal exactly when their dot product is zero.

## Badminton movement example

Consider two idealized movement vectors on a badminton court:

$$
\mathbf{p}=(3,4,-1),
\qquad
\mathbf{q}=(4,-3,0).
$$

Because

$$
\mathbf{p}\cdot\mathbf{q}=3(4)+4(-3)+(-1)(0)=0,
$$

the vectors are orthogonal.

## Cauchy-Schwarz, triangle inequality, and Pythagorean theorem

For vectors $\mathbf{u}$ and $\mathbf{v}$:

$$
|\mathbf{u}\cdot\mathbf{v}|\le \|\mathbf{u}\|\|\mathbf{v}\|,
$$

$$
\|\mathbf{u}+\mathbf{v}\|\le \|\mathbf{u}\|+\|\mathbf{v}\|.
$$

If $\mathbf{u}\perp\mathbf{v}$, then

$$
\|\mathbf{u}+\mathbf{v}\|^2=\|\mathbf{u}\|^2+\|\mathbf{v}\|^2.
$$

In [ ]:
p=sp.Matrix([3,4,-1]); q=sp.Matrix([4,-3,0])
left=sp.simplify((p+q).dot(p+q))
right=sp.simplify(p.dot(p)+q.dot(q))
print('p dot q =',p.dot(q))
print('||p+q||^2 =',left,'and ||p||^2+||q||^2 =',right)

## Matrix form of the dot product

Treat vectors as columns. Then

$$
\mathbf{u}\cdot\mathbf{v}=\mathbf{u}^{T}\mathbf{v}.
$$

For example,

$$
\begin{bmatrix}4&1&-2\end{bmatrix}
\begin{bmatrix}2\\-3\\5\end{bmatrix}
=-5.
$$

> **Closing note**
>
> The dot product connects algebra and geometry: it measures directional alignment, determines angles, and detects orthogonality.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Norm or length | $\|\mathbf{v}\|=\sqrt{\mathbf{v}\cdot\mathbf{v}}$. |
| Unit vector | Vector of length $1$. |
| Distance | $\|\mathbf{u}-\mathbf{v}\|$. |
| Dot product | Sum of componentwise products. |
| Orthogonal | Dot product equals zero. |
| Angle | Determined by the normalized dot product. |
| Cauchy-Schwarz inequality | $|\mathbf{u}\cdot\mathbf{v}|\le\|\mathbf{u}\|\|\mathbf{v}\|$. |
| Triangle inequality | $\|\mathbf{u}+\mathbf{v}\|\le\|\mathbf{u}\|+\|\mathbf{v}\|$. |

## Section 5.1 worksheet

1. Find the norm and a unit vector in the direction of $(6,-3,6)$.
2. Find the distance between $(1,4,-2)$ and $(5,-2,1)$.
3. Compute the dot product and angle between $(1,2,2)$ and $(2,0,-1)$.
4. Determine whether $(2,-1,4)$ and $(1,6,1)$ are orthogonal.
5. Verify Cauchy-Schwarz for $(2,1)$ and $(-1,3)$.
6. Write the dot product of $(a,b,c)$ and $(x,y,z)$ as a matrix product.

# Section 5.2 - Inner Product Spaces

## Learning objectives

Students should be able to:

- verify the inner-product axioms;
- compute inner products in vector, matrix, polynomial, and function spaces;
- use induced length, distance, angle, and orthogonality;
- find orthogonal projections in general inner product spaces.

### Definition: inner product

> An inner product $\langle\mathbf{u},\mathbf{v}\rangle$ is symmetric, linear in one argument, and positive definite. A vector space equipped with an inner product is an inner product space.

## The four real inner-product axioms

For all $\mathbf{u},\mathbf{v},\mathbf{w}$ and scalar $c$:

$$
\langle\mathbf{u},\mathbf{v}\rangle
=\langle\mathbf{v},\mathbf{u}\rangle,
$$

$$
\langle\mathbf{u}+\mathbf{v},\mathbf{w}\rangle
=\langle\mathbf{u},\mathbf{w}\rangle+\langle\mathbf{v},\mathbf{w}\rangle,
$$

$$
\langle c\mathbf{u},\mathbf{v}\rangle
=c\langle\mathbf{u},\mathbf{v}\rangle,
$$

and

$$
\langle\mathbf{v},\mathbf{v}\rangle\ge0,
$$

with equality only for $\mathbf{v}=\mathbf{0}$.

A proposed inner product must satisfy **all four** axioms. One failed axiom is enough to reject the operation.


## Weighted R&B feature inner product

Suppose a fictional song feature vector is $(r,b)$, where the two components represent two normalized audio features. Define

$$
\langle\mathbf{u},\mathbf{v}\rangle_W
=2u_1v_1+5u_2v_2.
$$

The positive weights make both coordinate contributions nonnegative in

$$
\langle\mathbf{v},\mathbf{v}\rangle_W
=2v_1^2+5v_2^2.
$$

For

$$
\mathbf{u}=(3,-1),
\qquad
\mathbf{v}=(2,4),
$$

we obtain

$$
\langle\mathbf{u},\mathbf{v}\rangle_W
=2(3)(2)+5(-1)(4)=-8.
$$

The induced norm of $\mathbf{u}$ is

$$
\|\mathbf{u}\|_W=\sqrt{2(3)^2+5(-1)^2}=\sqrt{23}.
$$

For a weighted dot product, the weights should be positive. Zero or negative weights can destroy positive definiteness.


> **A form that fails**
>
> The expression $u_1v_1-u_2v_2$ is not an inner product on $\mathbb{R}^2$, because $\langle(0,1),(0,1)\rangle=-1<0$.

## Inner product on polynomial coefficient vectors

For

$$
p(x)=a_0+a_1x+a_2x^2,
\qquad
q(x)=b_0+b_1x+b_2x^2,
$$

define

$$
\langle p,q\rangle
=a_0b_0+2a_1b_1+3a_2b_2.
$$

Let

$$
p(x)=1+2x-x^2,
\qquad
q(x)=3-x+2x^2.
$$

Then

$$
\langle p,q\rangle
=1(3)+2(2)(-1)+3(-1)(2)
=-7.
$$

In [ ]:
x=sp.symbols('x')
p=1+2*x-x**2; q=3-x+2*x**2
cp=sp.Matrix([1,2,-1]); cq=sp.Matrix([3,-1,2])
W=sp.diag(1,2,3)
print('Weighted coefficient inner product:',(cp.T*W*cq)[0])

## Integral inner product on functions

On $C[a,b]$, a common inner product is

$$
\langle f,g\rangle=\int_a^b f(x)g(x)\,dx.
$$

For $f(x)=1$ and $g(x)=x$ on $[0,1]$,

$$
\langle f,g\rangle=\int_0^1x\,dx=\frac12.
$$

> **Projection formula**
>
> For $\mathbf{v}\ne\mathbf{0}$, $\operatorname{proj}_{\mathbf{v}}\mathbf{u}=\dfrac{\langle\mathbf{u},\mathbf{v}\rangle}{\langle\mathbf{v},\mathbf{v}\rangle}\mathbf{v}$.

## Orthogonal projection in $\mathbb{R}^3$

Let

$$
\mathbf{u}=(6,3,3),
\qquad
\mathbf{v}=(1,2,2).
$$

Since

$$
\mathbf{u}\cdot\mathbf{v}=18,
\qquad
\mathbf{v}\cdot\mathbf{v}=9,
$$

we have

$$
\operatorname{proj}_{\mathbf{v}}\mathbf{u}
=\frac{18}{9}\mathbf{v}
=2(1,2,2)
=(2,4,4).
$$

The orthogonal residual is

$$
\mathbf{u}-\operatorname{proj}_{\mathbf{v}}\mathbf{u}
=(4,-1,-1),
$$

and

$$
(4,-1,-1)\cdot(1,2,2)=0.
$$

In [ ]:
u=sp.Matrix([6,3,3]); v=sp.Matrix([1,2,2])
proj=projection(u,v); residual=u-proj
show_vector(proj,r'\operatorname{proj}_{\mathbf{v}}\mathbf{u}')
show_vector(residual,r'\mathbf{u}-\operatorname{proj}_{\mathbf{v}}\mathbf{u}')
print('Residual dot v =',residual.dot(v))
plot_vectors_3d([u,v,proj,residual],['u','v','proj','residual'],'Projection and orthogonal residual')

## Projection minimizes distance

Every vector of the form $c\mathbf{v}$ lies on the line spanned by $\mathbf{v}$. The projection is the unique point on that line closest to $\mathbf{u}$.

> **Closing note**
>
> An inner product equips an abstract vector space with geometry: length, angle, orthogonality, and projection.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Inner product | Scalar-valued function satisfying symmetry, linearity, and positive definiteness. |
| Inner product space | Vector space with a specified inner product. |
| Induced norm | $\|\mathbf{v}\|=\sqrt{\langle\mathbf{v},\mathbf{v}\rangle}$. |
| Orthogonal | Inner product equals zero. |
| Weighted inner product | Coordinates contribute with positive weights. |
| Function inner product | Often an integral of a product. |
| Orthogonal projection | Closest vector in a one-dimensional subspace. |
| Residual | Difference between a vector and its projection. |

## Section 5.2 worksheet

1. For $\langle u,v\rangle=3u_1v_1+2u_2v_2$, find $\langle(2,-1),(4,3)\rangle$ and the induced norm of $(2,-1)$.
2. Explain why $u_1v_1+u_2v_2-u_3v_3$ is not an inner product on $\mathbb{R}^3$.
3. Using coefficient inner product, find $\langle1+x+2x^2,2-3x+x^2\rangle$.
4. On $C[0,1]$, compute $\langle1,x^2\rangle=\int_0^1x^2\,dx$.
5. Find the projection of $(5,1,4)$ onto $(1,0,2)$.
6. Verify that the residual in Problem 5 is orthogonal to the projection direction.

# Section 5.3 - Orthonormal Bases and the Gram-Schmidt Process

## Learning objectives

Students should be able to:

- distinguish orthogonal and orthonormal sets;
- prove that an orthogonal nonzero set is linearly independent;
- find coordinates relative to an orthonormal basis;
- apply the Gram-Schmidt orthonormalization process;
- construct an orthonormal basis for a solution space.

### Orthogonal and orthonormal sets

> A set is orthogonal when distinct vectors have inner product zero. It is orthonormal when it is orthogonal and every vector has norm one.

## A nonstandard orthonormal basis for $\mathbb{R}^3$

Consider

$$
\mathbf{u}_1=\frac1{\sqrt2}(1,1,0),
\qquad
\mathbf{u}_2=\frac1{\sqrt2}(1,-1,0),
\qquad
\mathbf{u}_3=(0,0,1).
$$

The pairwise dot products are zero, and each vector has norm one. Therefore the set is an orthonormal basis of $\mathbb{R}^3$.

Verify explicitly that all pairwise dot products are zero and that

$$
\|\mathbf u_1\|=\|\mathbf u_2\|=\|\mathbf u_3\|=1.
$$


## Coordinates relative to an orthonormal basis

For

$$
\mathbf{w}=(9,12,20),
$$

the coordinates are simply the inner products:

$$
\begin{aligned}
c_1&=\mathbf{w}\cdot\mathbf{u}_1=\frac{21}{\sqrt2},\\
c_2&=\mathbf{w}\cdot\mathbf{u}_2=-\frac3{\sqrt2},\\
c_3&=\mathbf{w}\cdot\mathbf{u}_3=20.
\end{aligned}
$$

Thus

$$
[\mathbf{w}]_B=
\begin{bmatrix}
21/\sqrt2\\
-3/\sqrt2\\
20
\end{bmatrix}.
$$

In [ ]:
s2=sp.sqrt(2)
U=[sp.Matrix([1/s2,1/s2,0]),sp.Matrix([1/s2,-1/s2,0]),sp.Matrix([0,0,1])]
w=sp.Matrix([9,12,20])
coords=sp.Matrix([sp.simplify(w.dot(ui)) for ui in U])
show_vector(coords,r'[\mathbf{w}]_B')
print('Reconstruction check:',sp.simplify(sum((coords[i]*U[i] for i in range(3)),sp.zeros(3,1))-w))

> **Gram-Schmidt idea**
>
> Subtract previously constructed projections to create orthogonal vectors, then normalize each vector.

**Goal:** start with a linearly independent set and produce an orthonormal basis for the same span. Remove projections onto directions already constructed, then normalize.


## Detailed Gram-Schmidt example in $\mathbb{R}^3$

Start with

$$
\mathbf{v}_1=(1,1,0),
\qquad
\mathbf{v}_2=(1,0,1),
\qquad
\mathbf{v}_3=(0,1,1).
$$

First,

$$
\mathbf{w}_1=\mathbf{v}_1=(1,1,0).
$$

For the second vector,

$$
\operatorname{proj}_{\mathbf{w}_1}\mathbf{v}_2
=\frac{\mathbf{v}_2\cdot\mathbf{w}_1}{\mathbf{w}_1\cdot\mathbf{w}_1}\mathbf{w}_1
=\frac12(1,1,0),
$$

so

$$
\mathbf{w}_2
=\mathbf{v}_2-\operatorname{proj}_{\mathbf{w}_1}\mathbf{v}_2
=\left(\frac12,-\frac12,1\right).
$$

Scaling does not change orthogonality, so use the simpler vector

$$
\mathbf{w}_2=(1,-1,2).
$$

For the third vector,

$$
\operatorname{proj}_{\mathbf{w}_1}\mathbf{v}_3
=\frac12(1,1,0),
$$

and

$$
\operatorname{proj}_{\mathbf{w}_2}\mathbf{v}_3
=\frac16(1,-1,2).
$$

Therefore

$$
\mathbf{v}_3
-\operatorname{proj}_{\mathbf{w}_1}\mathbf{v}_3
-\operatorname{proj}_{\mathbf{w}_2}\mathbf{v}_3
=\frac23(-1,1,1).
$$

Use

$$
\mathbf{w}_3=(-1,1,1).
$$

After normalization,

$$
\boxed{
\mathbf{u}_1=\frac1{\sqrt2}(1,1,0),
\quad
\mathbf{u}_2=\frac1{\sqrt6}(1,-1,2),
\quad
\mathbf{u}_3=\frac1{\sqrt3}(-1,1,1)
}.
$$

In [ ]:
V=[sp.Matrix([1,1,0]),sp.Matrix([1,0,1]),sp.Matrix([0,1,1])]
Ws,Us=gram_schmidt_detailed(V)
print('Orthonormality matrix:')
G=sp.Matrix([[sp.simplify(ui.dot(uj)) for uj in Us] for ui in Us])
display(G)
plot_vectors_3d(Us,['u1','u2','u3'],'Orthonormal basis produced by Gram-Schmidt')

## Gram-Schmidt on a solution space

Consider

$$
-x_1+x_2+x_3=0,
\qquad
x_1+x_4=0.
$$

Solving gives a basis

$$
\mathbf{v}_1=(1,1,0,-1),
\qquad
\mathbf{v}_2=(1,0,1,-1).
$$

Normalize the first vector:

$$
\mathbf{u}_1=\frac1{\sqrt3}(1,1,0,-1).
$$

Remove the component of $\mathbf{v}_2$ in the direction of $\mathbf{v}_1$:

$$
\mathbf{w}_2
=\mathbf{v}_2-\frac{\mathbf{v}_2\cdot\mathbf{v}_1}{\mathbf{v}_1\cdot\mathbf{v}_1}\mathbf{v}_1
=\mathbf{v}_2-\frac23\mathbf{v}_1
=\frac13(1,-2,3,-1).
$$

Use the scaled vector $(1,-2,3,-1)$ and normalize:

$$
\mathbf{u}_2=\frac1{\sqrt{15}}(1,-2,3,-1).
$$

Thus an orthonormal basis for the solution space is

$$
\left\{
\frac1{\sqrt3}(1,1,0,-1),
\frac1{\sqrt{15}}(1,-2,3,-1)
\right\}.
$$

The “solution space” here is the **nullspace** of the coefficient matrix: the set of vectors satisfying $A\mathbf x=\mathbf0$. We first find a nullspace basis and then orthonormalize it.


In [ ]:
A=sp.Matrix([[-1,1,1,0],[1,0,0,1]])
print('Nullspace basis:')
for z in A.nullspace(): display(z)
U1=sp.Matrix([1,1,0,-1])/sp.sqrt(3)
U2=sp.Matrix([1,-2,3,-1])/sp.sqrt(15)
print('A*u1 =',A*U1,'A*u2 =',A*U2,'u1 dot u2 =',sp.simplify(U1.dot(U2)))

> **Closing note**
>
> Orthonormal bases simplify coordinates and projections because coefficient extraction requires only inner products.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Orthogonal set | Distinct vectors have inner product zero. |
| Orthonormal set | Orthogonal set of unit vectors. |
| Orthogonal basis | Basis whose vectors are mutually orthogonal. |
| Fourier coefficient | Coordinate obtained by inner product with an orthonormal basis vector. |
| Gram-Schmidt process | Converts a basis into an orthogonal or orthonormal basis. |
| Normalization | Divide a nonzero vector by its norm. |
| Projection subtraction | Removes components along earlier basis vectors. |

## Section 5.3 worksheet

1. Determine whether $\{(1,1,0),(1,-1,0),(0,0,2)\}$ is orthogonal. Is it orthonormal?
2. Normalize the set in Problem 1.
3. Find the coordinates of $(4,2,3)$ relative to the normalized basis from Problem 2.
4. Apply Gram-Schmidt to $(1,0)$ and $(1,2)$.
5. Apply Gram-Schmidt to $(1,1,0)$ and $(1,0,1)$.
6. Find an orthonormal basis for $\operatorname{span}\{(1,1,1),(1,-1,0)\}$.

# Section 5.4 - Mathematical Models and Least Squares Analysis

## Learning objectives

Students should be able to:

- formulate a least-squares problem;
- find orthogonal complements and direct sums;
- project onto a subspace;
- identify the four fundamental subspaces;
- solve normal equations;
- construct linear and quadratic least-squares models.

> **Least-squares problem**
>
> Given $A\in\mathbb{R}^{m\times n}$ and $\mathbf{b}\in\mathbb{R}^m$, find $\widehat{\mathbf{x}}$ minimizing $\|A\mathbf{x}-\mathbf{b}\|$.

> **Orthogonal complement**
>
> For a subspace $S\subseteq\mathbb{R}^n$, $S^\perp$ is the set of vectors orthogonal to every vector in $S$.

## Orthogonal complement example

Let

$$
S=\operatorname{span}\{(1,1,0,1),(1,0,1,1)\}.
$$

A vector $\mathbf{x}$ lies in $S^\perp$ when

$$
\begin{bmatrix}
1&1&0&1\\
1&0&1&1
\end{bmatrix}
\mathbf{x}=\mathbf{0}.
$$

A basis for the nullspace is

$$
\left\{
(-1,1,1,0),
(-1,0,0,1)
\right\}.
$$

Therefore,

$$
S^\perp
=\operatorname{span}\{(-1,1,1,0),(-1,0,0,1)\}.
$$

In [ ]:
Smat=sp.Matrix([[1,1,0,1],[1,0,1,1]])
for i,z in enumerate(Smat.nullspace(),1): show_vector(z,rf'\mathbf{{z}}_{i}')

> **Projection onto a subspace**
>
> If $\{\mathbf{u}_1,\ldots,\mathbf{u}_k\}$ is an orthonormal basis for $S$, then $\operatorname{proj}_S\mathbf{v}=\sum_i\langle\mathbf{v},\mathbf{u}_i\rangle\mathbf{u}_i$. For an orthogonal basis, divide each term by $\mathbf{w}_i\cdot\mathbf{w}_i$.

## Projection onto a plane

Let

$$
\mathbf{w}_1=(1,1,0),
\qquad
\mathbf{w}_2=(1,-1,2).
$$

These vectors are orthogonal. Project

$$
\mathbf{v}=(9,12,20)
$$

onto $S=\operatorname{span}\{\mathbf{w}_1,\mathbf{w}_2\}$:

$$
\operatorname{proj}_S\mathbf{v}
=\frac{21}{2}\mathbf{w}_1+\frac{37}{6}\mathbf{w}_2
=\left(\frac{50}{3},\frac{13}{3},\frac{37}{3}\right).
$$

The residual is

$$
\mathbf{v}-\operatorname{proj}_S\mathbf{v}
=\frac{23}{3}(-1,1,1),
$$

which is orthogonal to both spanning vectors.

In [ ]:
v=sp.Matrix([9,12,20]); w1=sp.Matrix([1,1,0]); w2=sp.Matrix([1,-1,2])
projS=sp.simplify(v.dot(w1)/w1.dot(w1)*w1 + v.dot(w2)/w2.dot(w2)*w2)
res=v-projS
show_vector(projS,r'\operatorname{proj}_{S}\mathbf{v}')
show_vector(res,r'\mathbf{r}')
print('r dot w1 =',sp.simplify(res.dot(w1)),'r dot w2 =',sp.simplify(res.dot(w2)))

## The four fundamental subspaces

For

$$
A=
\begin{bmatrix}
1&2&0&1\\
0&1&1&1\\
1&3&1&2
\end{bmatrix},
$$

the third row equals the sum of the first two. The RREF is

$$
\begin{bmatrix}
1&0&-2&-1\\
0&1&1&1\\
0&0&0&0
\end{bmatrix}.
$$

Therefore:

- $\operatorname{rank}(A)=2$;
- a row-space basis is $\{(1,0,-2,-1),(0,1,1,1)\}$;
- a column-space basis is formed from the first two original columns;
- a nullspace basis is $\{(2,-1,1,0),(1,-1,0,1)\}$;
- a basis for $N(A^T)$ is $\{(-1,-1,1)\}$.

### Formal overview

For an $m\times n$ matrix $A$:

$$
\operatorname{Col}(A)\subseteq\mathbb R^m,\qquad
\operatorname{Null}(A)\subseteq\mathbb R^n,
$$

$$
\operatorname{Row}(A)=\operatorname{Col}(A^T)\subseteq\mathbb R^n,
\qquad
\operatorname{Null}(A^T)\subseteq\mathbb R^m.
$$

Moreover,

$$
\operatorname{Row}(A)^\perp=\operatorname{Null}(A),
\qquad
\operatorname{Col}(A)^\perp=\operatorname{Null}(A^T).
$$


In [ ]:
A=sp.Matrix([[1,2,0,1],[0,1,1,1],[1,3,1,2]])
R,piv=A.rref()
show_matrix(R,r'\operatorname{rref}(A)')
print('Pivots:',piv)
print('Nullspace:')
for z in A.nullspace(): display(z)
print('Left nullspace:')
for z in A.T.nullspace(): display(z)

> **Normal equations**
>
> A least-squares solution satisfies $A^TA\widehat{\mathbf{x}}=A^T\mathbf{b}$. The fitted vector $A\widehat{\mathbf{x}}$ is the projection of $\mathbf{b}$ onto $\operatorname{Col}(A)$.

## Driving-data regression line

A fictional driving dataset uses

$$
(1,2),(2,3),(3,5),(4,4),(5,7).
$$

Fit

$$
y=c_0+c_1x.
$$

The design matrix and data vector are

$$
A=
\begin{bmatrix}
1&1\\
1&2\\
1&3\\
1&4\\
1&5
\end{bmatrix},
\qquad
\mathbf{b}=
\begin{bmatrix}2\\3\\5\\4\\7\end{bmatrix}.
$$

The normal equations are

$$
A^TA
\begin{bmatrix}c_0\\c_1\end{bmatrix}
=A^T\mathbf{b}.
$$

Compute

$$
A^TA=
\begin{bmatrix}5&15\\15&55\end{bmatrix},
\qquad
A^T\mathbf{b}=
\begin{bmatrix}21\\75\end{bmatrix}.
$$

Solving gives

$$
c_0=\frac9{10},
\qquad
c_1=\frac{11}{10}.
$$

Thus

$$
\widehat y=0.9+1.1x.
$$

In [ ]:
xs=np.array([1,2,3,4,5],dtype=float)
ys=np.array([2,3,5,4,7],dtype=float)
A=np.column_stack([np.ones_like(xs),xs])
coef=np.linalg.solve(A.T@A,A.T@ys)
print('coefficients:',coef)
xx=np.linspace(.5,5.5,200)
plt.figure(figsize=(7,5)); plt.scatter(xs,ys,label='data'); plt.plot(xx,coef[0]+coef[1]*xx,label='least-squares line')
plt.xlabel('x'); plt.ylabel('y'); plt.grid(alpha=.3); plt.legend(); plt.title('Fictional driving-data regression'); plt.show()

## Quadratic travel model

Suppose a fictional travel-planning response is observed at

$$
(0,3),(1,6),(2,11),(3,18),(4,28).
$$

Fit

$$
y=c_0+c_1x+c_2x^2.
$$

Solving the quadratic normal equations gives

$$
\boxed{
\widehat y
=\frac{108}{35}+\frac{57}{35}x+\frac87x^2
}.
$$

The residual sum of squares is

$$
\frac4{35}.
$$

In [ ]:
xs2=np.arange(5,dtype=float); ys2=np.array([3,6,11,18,28],dtype=float)
A2=np.column_stack([np.ones_like(xs2),xs2,xs2**2])
c2=np.linalg.solve(A2.T@A2,A2.T@ys2)
print('quadratic coefficients:',c2)
xx=np.linspace(-.2,4.2,200)
plt.figure(figsize=(7,5)); plt.scatter(xs2,ys2,label='data'); plt.plot(xx,c2[0]+c2[1]*xx+c2[2]*xx**2,label='quadratic fit')
plt.grid(alpha=.3); plt.legend(); plt.title('Fictional travel-planning quadratic model'); plt.show()

> **Closing note**
>
> Least squares replaces an inconsistent equation by an orthogonal projection problem. The residual must be orthogonal to the column space.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Least-squares solution | Minimizes $\|A\mathbf{x}-\mathbf{b}\|$. |
| Orthogonal complement | Vectors orthogonal to every vector in a subspace. |
| Direct sum | Unique decomposition into vectors from two subspaces. |
| Projection onto a subspace | Closest vector in that subspace. |
| Four fundamental subspaces | $\operatorname{Col}(A)$, $N(A)$, $\operatorname{Col}(A^T)$, and $N(A^T)$. |
| Normal equations | $A^TA\widehat{\mathbf{x}}=A^T\mathbf{b}$. |
| Residual | $\mathbf{b}-A\widehat{\mathbf{x}}$. |
| Regression model | Least-squares model for observed data. |

## Section 5.4 worksheet

1. Find a basis for the orthogonal complement of $\operatorname{span}\{(1,1,0),(0,1,1)\}$.
2. Project $(4,5,6)$ onto the line spanned by $(1,1,1)$.
3. Project $(3,1,4)$ onto the plane spanned by the orthogonal vectors $(1,1,0)$ and $(1,-1,2)$.
4. Find rank, a row-space basis, a column-space basis, and a nullspace basis for $\begin{bmatrix}1&2&3\\2&4&6\\0&1&1\end{bmatrix}$.
5. Find the least-squares line for $(0,1),(1,2),(2,2),(3,5)$.
6. Verify that the residual from Problem 5 is orthogonal to both columns of the design matrix.

# Section 5.5 - Applications of Inner Product Spaces

## Learning objectives

Students should be able to:

- compute and interpret the cross product in $\mathbb{R}^3$;
- find vectors orthogonal to two given vectors;
- use cross products to find area and angle;
- construct continuous least-squares approximations;
- compute finite Fourier approximations.

### Definition: cross product

> For $\mathbf{u},\mathbf{v}\in\mathbb{R}^3$, the vector $\mathbf{u}\times\mathbf{v}$ is orthogonal to both vectors and has magnitude $\|\mathbf{u}\|\|\mathbf{v}\|\sin\theta$.

## Hiking-trail cross product

Let two trail-direction vectors be

$$
\mathbf{u}=(4,1,2),
\qquad
\mathbf{v}=(1,3,-1).
$$

Then

$$
\mathbf{u}\times\mathbf{v}
=
\begin{vmatrix}
\mathbf{i}&\mathbf{j}&\mathbf{k}\\
4&1&2\\
1&3&-1
\end{vmatrix}
=(-7,6,11).
$$

Check orthogonality:

$$
(-7,6,11)\cdot(4,1,2)=0,
$$

$$
(-7,6,11)\cdot(1,3,-1)=0.
$$

The parallelogram area is

$$
\|\mathbf{u}\times\mathbf{v}\|=\sqrt{206}.
$$

The triangle area is

$$
\frac{\sqrt{206}}2.
$$

In [ ]:
u=sp.Matrix([4,1,2]); v=sp.Matrix([1,3,-1]); cross=u.cross(v)
show_vector(cross,r'\mathbf{u}\times\mathbf{v}')
show_exact(sp.sqrt(cross.dot(cross)),r'\|\mathbf{u}\times\mathbf{v}\|')
print('dot checks:',cross.dot(u),cross.dot(v))
plot_vectors_3d([u,v,cross],['u','v','u x v'],'Cross product orthogonal to two vectors')

## Unit normal vectors

A unit vector orthogonal to both $\mathbf{u}$ and $\mathbf{v}$ is

$$
\frac1{\sqrt{206}}(-7,6,11).
$$

Its negative is the other unit normal.

> **Continuous least squares**
>
> For an inner product $\langle f,g\rangle=\int_a^b f(x)g(x)\,dx$, the least-squares approximation is the orthogonal projection of $f$ onto a finite-dimensional function subspace.

## Best linear approximation to $f(x)=x^2$ on $[0,1]$

Approximate $f(x)=x^2$ by

$$
g(x)=a+bx.
$$

The error must be orthogonal to both basis functions $1$ and $x$:

$$
\langle x^2-a-bx,1\rangle=0,
$$

$$
\langle x^2-a-bx,x\rangle=0.
$$

Therefore,

$$
\begin{aligned}
\int_0^1(x^2-a-bx)\,dx&=0,\\
\int_0^1x(x^2-a-bx)\,dx&=0.
\end{aligned}
$$

This gives

$$
\begin{aligned}
a+\frac12b&=\frac13,\\
\frac12a+\frac13b&=\frac14.
\end{aligned}
$$

Solving,

$$
a=-\frac16,
\qquad
b=1.
$$

Hence

$$
\boxed{g(x)=x-\frac16}.
$$

In [ ]:
x=sp.symbols('x', real=True)
G=sp.Matrix([[sp.integrate(1,(x,0,1)),sp.integrate(x,(x,0,1))],
             [sp.integrate(x,(x,0,1)),sp.integrate(x**2,(x,0,1))]])
rhs=sp.Matrix([sp.integrate(x**2,(x,0,1)),sp.integrate(x**3,(x,0,1))])
coef=G.LUsolve(rhs)
show_matrix(G,'G'); show_vector(rhs,r'\mathbf{d}'); show_vector(coef,r'\begin{bmatrix}a\\b\end{bmatrix}')
xx=np.linspace(0,1,300)
plt.figure(figsize=(7,5)); plt.plot(xx,xx**2,label='$x^2$'); plt.plot(xx,xx-1/6,label='$x-1/6$')
plt.grid(alpha=.3); plt.legend(); plt.title('Continuous least-squares approximation'); plt.show()

## Fourier approximation

On $[0,2\pi]$, consider the centered ramp

$$
f(x)=x-\pi.
$$

Its Fourier coefficients satisfy

$$
a_0=0,
\qquad
a_n=0,
\qquad
b_n=-\frac2n.
$$

The third-order approximation is

$$
\boxed{
S_3(x)
=-2\sin x-\sin(2x)-\frac23\sin(3x)
}.
$$

In [ ]:
xx=np.linspace(0,2*np.pi,1000)
f=xx-np.pi
S1=-2*np.sin(xx)
S3=-2*np.sin(xx)-np.sin(2*xx)-(2/3)*np.sin(3*xx)
S10=sum((-2/n)*np.sin(n*xx) for n in range(1,11))
plt.figure(figsize=(8,5)); plt.plot(xx,f,label='$f(x)=x-\\pi$'); plt.plot(xx,S1,label='order 1'); plt.plot(xx,S3,label='order 3'); plt.plot(xx,S10,label='order 10')
plt.grid(alpha=.3); plt.legend(); plt.title('Fourier approximations of a centered ramp'); plt.show()

## Interactive Fourier order

In [ ]:
if WIDGETS_AVAILABLE:
    @interact(n=IntSlider(value=3,min=1,max=20,step=1,description='order'))
    def fourier_order(n=3):
        xx=np.linspace(0,2*np.pi,1000)
        f=xx-np.pi
        Sn=sum((-2/k)*np.sin(k*xx) for k in range(1,n+1))
        plt.figure(figsize=(8,4.8)); plt.plot(xx,f,label='target'); plt.plot(xx,Sn,label=f'order {n}')
        plt.grid(alpha=.3); plt.legend(); plt.ylim(-4,4); plt.show()
else:
    print('ipywidgets is unavailable; the static Fourier plots above remain usable.')

> **Closing note**
>
> Cross products solve three-dimensional perpendicularity problems, while function-space projections create best approximations and Fourier models.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Cross product | Vector orthogonal to two vectors in $\mathbb{R}^3$. |
| Unit normal | Unit vector perpendicular to a plane. |
| Parallelogram area | $\|\mathbf{u}\times\mathbf{v}\|$. |
| Continuous least squares | Projection in a function inner product space. |
| Approximating subspace | Finite-dimensional function space used for approximation. |
| Fourier coefficients | Coordinates relative to trigonometric orthogonal functions. |
| Fourier approximation | Finite trigonometric least-squares approximation. |
| Approximation order | Highest sine or cosine frequency retained. |

## Section 5.5 worksheet

1. Find $(2,1,-1)\times(1,-2,3)$.
2. Verify that the answer to Problem 1 is orthogonal to both original vectors.
3. Find the parallelogram and triangle areas determined by the two vectors in Problem 1.
4. Find a unit normal to the plane spanned by $(1,0,1)$ and $(0,2,1)$.
5. Find the best constant approximation to $f(x)=x$ on $[0,2]$.
6. Write the second-order Fourier approximation of $f(x)=x-\pi$ on $[0,2\pi]$.

# Chapter 5 synthesis

Inner products turn vector spaces into geometric environments. Once length and orthogonality are available, we can build orthonormal coordinates, projections, least-squares models, cross-product geometry, and Fourier approximations.

## Core connections

$$
\text{dot product}
\longrightarrow
\text{orthogonality}
\longrightarrow
\text{projection}
\longrightarrow
\text{least squares}.
$$

$$
\text{basis}
\longrightarrow
\text{Gram-Schmidt}
\longrightarrow
\text{orthonormal basis}
\longrightarrow
\text{simple coordinates and approximation}.
$$